In [ ]:
# version 7 2025.12.04 ~ 05

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
import re
import warnings
from tqdm import tqdm
from gensim.models import FastText
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

from pycaret.regression import *
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


class MercariPyCaretAnalyzer7:
    """
    Mercari Price Suggestion - 개선 버전
    - TF-IDF / FastText / BERT 벡터화 선택 가능
    - Stratified undersampling (35%)
    - 희귀값 통합, 추가 피처 생성
    """

    # __init__ ##############################
    def __init__(self,
                 data_dir="../data",
                 images_dir="../images",
                 results_dir="../results"):
        self.data_dir = data_dir
        self.images_dir = images_dir
        self.results_dir = results_dir

        self.train = None
        self.test = None
        self.best_model = None
        self.setup_result = None
        self.metrics = {}

        os.makedirs(self.images_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)
    # eof -----------------------------------

    # _collapse_rare_values ##################
    def _collapse_rare_values(self, col, top_k, rare_label="Other"):
        """희귀값 통합"""
        combined = pd.concat([self.train[col], self.test[col]], axis=0)
        value_counts = combined.value_counts()
        top_values = set(value_counts.index[:top_k])
        self.train[col] = self.train[col].apply(lambda x: x if x in top_values else rare_label)
        self.test[col] = self.test[col].apply(lambda x: x if x in top_values else rare_label)
        gc.collect()
    # eof -----------------------------------

    # _simple_normalize #####################
    def _simple_normalize(self, text: str) -> str:
        """텍스트 정규화"""
        text = str(text).lower()
        text = re.sub(r"[_\-\./]", " ", text)
        text = re.sub(r"\d+", " num ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text
    # eof -----------------------------------

    # _stratified_sample ####################
    def _stratified_sample(self, frac=0.35, bins=10):
        """층화 샘플링"""
        self.train["price_bin"] = pd.qcut(self.train["price"], q=bins, duplicates="drop")
        sampled = self.train.groupby("price_bin", group_keys=False).apply(
            lambda x: x.sample(frac=frac, random_state=23)
        )
        self.train = sampled.drop(columns=["price_bin"]).reset_index(drop=True)
        gc.collect()
        print(f"⚠️ Stratified undersampling 적용: train {self.train.shape}")
    # eof -----------------------------------

    # load_data #############################
    def load_data(self, train_file="train.tsv", test_file="test.tsv", sep="\t", undersample_frac=0.35):
        """데이터 로딩 및 전처리"""
        print("📂 데이터 로딩 시작...")
        train_path = os.path.join(self.data_dir, train_file)
        test_path = os.path.join(self.data_dir, test_file)

        self.train = pd.read_csv(train_path, sep=sep)
        self.test = pd.read_csv(test_path, sep=sep)

        # price 로그 변환
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])
        self.train["price"] = np.log1p(self.train["price"])

        # 층화 언더샘플링
        if undersample_frac is not None:
            self._stratified_sample(frac=undersample_frac)

        # category split + 결측치 처리
        for df in [self.train, self.test]:
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: (x.split("/") if isinstance(x, str) and "/" in x else ["missing"]*3)
                )
            )
            df["brand_name"] = df["brand_name"].fillna("Unknown").astype(str)
            df["item_description"] = df["item_description"].fillna("No description").astype(str)
            df["name"] = df["name"].fillna("No name").astype(str)
            df.drop(columns=["category_name"], inplace=True)

        # 희귀값 통합
        print("🔄 희귀값 통합 중...")
        self._collapse_rare_values("brand_name", top_k=5000, rare_label="Other_brand")
        self._collapse_rare_values("main_cat", top_k=1000, rare_label="Other_main")
        self._collapse_rare_values("sub_cat", top_k=1000, rare_label="Other_sub")
        self._collapse_rare_values("sub_sub_cat", top_k=1000, rare_label="Other_sub_sub")

        # 길이 피처
        for df in [self.train, self.test]:
            df["name_len_char"] = df["name"].str.len()
            df["name_len_word"] = df["name"].str.split().str.len()
            df["desc_len_char"] = df["item_description"].str.len()
            df["desc_len_word"] = df["item_description"].str.split().str.len()
            df["has_brand_in_name"] = df.apply(
                lambda row: 1 if row["brand_name"].lower() in row["name"].lower() else 0, axis=1
            )
            df["has_brand_in_desc"] = df.apply(
                lambda row: 1 if row["brand_name"].lower() in row["item_description"].lower() else 0, axis=1
            )

        # 범주형 변환
        for df in [self.train, self.test]:
            df["shipping"] = df["shipping"].astype("category")
            df["item_condition_id"] = df["item_condition_id"].astype("category")

        gc.collect()
        print(f"✅ 데이터 로드 완료: train {self.train.shape}, test {self.test.shape}")
    # eof -----------------------------------

    # vectorize_text_tfidf ##################
    def vectorize_text_tfidf(self, max_features_name=15000, max_features_desc=20000, n_components=150):
        """TF-IDF 벡터화"""
        print("🔍 TF-IDF 벡터화 시작...")
        vec_name = TfidfVectorizer(max_features=max_features_name, min_df=2, max_df=0.95)
        vec_desc = TfidfVectorizer(max_features=max_features_desc, min_df=2, max_df=0.95)
        name_train = vec_name.fit_transform(self.train["name"])
        desc_train = vec_desc.fit_transform(self.train["item_description"])
        name_test = vec_name.transform(self.test["name"])
        desc_test = vec_desc.transform(self.test["item_description"])
        svd = TruncatedSVD(n_components=n_components, random_state=23)
        train_vec = np.hstack([svd.fit_transform(name_train), svd.fit_transform(desc_train)])
        test_vec = np.hstack([svd.transform(name_test), svd.transform(desc_test)])
        self.train_vectorized = pd.DataFrame(train_vec)
        self.test_vectorized = pd.DataFrame(test_vec)
        print(f"✅ TF-IDF 벡터화 완료: train {self.train_vectorized.shape}, test {self.test_vectorized.shape}")
    # eof -----------------------------------

    # vectorize_text_fasttext ################
    def vectorize_text_fasttext(self, text_columns=["name", "item_description"],
                                fasttext_size=100, fasttext_window=5, fasttext_min_count=2):
        """FastText 벡터화"""
        print("🔍 FastText 벡터화 시작...")
        sentences = []
        for col in text_columns:
            self.train[col] = self.train[col].fillna("").astype(str)
            self.test[col] = self.test[col].fillna("").astype(str)
            sentences += [str(x).split() for x in pd.concat([self.train[col], self.test[col]])]
        ft_model = FastText(sentences, vector_size=fasttext_size, window=fasttext_window,
                            min_count=fasttext_min_count, sg=1)
        def get_vector(text):
            words = text.split()
            vectors = [ft_model.wv[w] for w in words if w in ft_model.wv]
            return np.mean(vectors, axis=0) if vectors else np.zeros(fasttext_size)
        train_features, test_features = [], []
        for col in text_columns:
            train_features.append(np.vstack(self.train[col].apply(get_vector)))
            test_features.append(np.vstack(self.test[col].apply(get_vector)))
        train_vec = np.hstack(train_features)
        test_vec = np.hstack(test_features)
        self.train_vectorized = pd.DataFrame
        self.train_vectorized = pd.DataFrame(train_vec)
        self.test_vectorized = pd.DataFrame(test_vec)
        print(f"✅ FastText 벡터화 완료: train {self.train_vectorized.shape}, test {self.test_vectorized.shape}")
    # eof -----------------------------------

    # vectorize_text_bert ###################
    def vectorize_text_bert(self, text_columns=["name", "item_description"], bert_model_name="all-MiniLM-L6-v2"):
        """BERT 문장 임베딩 벡터화"""
        print(f"🔍 BERT 임베딩 시작... (모델={bert_model_name})")
        bert_model = SentenceTransformer(bert_model_name)

        train_features, test_features = [], []
        for col in text_columns:
            self.train[col] = self.train[col].fillna("").astype(str)
            self.test[col] = self.test[col].fillna("").astype(str)

            train_emb = bert_model.encode(self.train[col].tolist(), show_progress_bar=True)
            test_emb = bert_model.encode(self.test[col].tolist(), show_progress_bar=True)

            train_features.append(train_emb)
            test_features.append(test_emb)

        train_vec = np.hstack(train_features)
        test_vec = np.hstack(test_features)

        self.train_vectorized = pd.DataFrame(train_vec)
        self.test_vectorized = pd.DataFrame(test_vec)
        print(f"✅ BERT 벡터화 완료: train {self.train_vectorized.shape}, test {self.test_vectorized.shape}")
    # eof -----------------------------------

    # vectorize_text ########################
    def vectorize_text(self, method="tfidf", **kwargs):
        """
        텍스트 벡터화 통합 인터페이스
        - 저장된 결과가 있으면 불러오고, 없으면 새로 계산 후 저장
        """
        if self.load_vectorized(method):
            return  # 이미 저장된 결과 불러옴

        if method == "tfidf":
            self.vectorize_text_tfidf(**kwargs)
        elif method == "fasttext":
            self.vectorize_text_fasttext(**kwargs)
        elif method == "bert":
            self.vectorize_text_bert(**kwargs)
        else:
            raise ValueError("method must be one of ['tfidf','fasttext','bert']")

        self.save_vectorized(method)
    # eof -----------------------------------
    

    # setup_pycaret #########################
    def setup_pycaret(self, session_id=23, fold=3, use_gpu=False):
        """PyCaret 환경 설정"""
        print("🔧 PyCaret setup 시작...")
        categorical_cols = ["main_cat","sub_cat","sub_sub_cat","brand_name","item_condition_id","shipping"]
        existing_categorical = [col for col in categorical_cols if col in self.train_vectorized.columns]

        self.setup_result = setup(
            data=self.train_vectorized.assign(price=self.train["price"].reset_index(drop=True)),
            target="price",
            session_id=session_id,
            categorical_features=existing_categorical if existing_categorical else None,
            normalize=True,
            transformation=False,
            fold_strategy="kfold",
            fold=fold,
            use_gpu=use_gpu,
            n_jobs=4,
            verbose=True,
            html=False
        )
        gc.collect()
        print("✅ PyCaret setup 완료")
    # eof -----------------------------------

    # find_and_blend_models #################
    def find_and_blend_models(self, top_n=3, sort_metric="R2", use_kaggle_winners=True, use_tqdm=True, include_autogluon=True):
        """상위권 모델 탐색 및 블렌딩"""
        if self.setup_result is None:
            raise ValueError("먼저 setup_pycaret()를 실행하세요.")

        if use_kaggle_winners:
            print("🏆 Mercari Kaggle 상위권 모델 학습")
            model_names = ["lightgbm","ridge","catboost","xgboost","et"]
            if include_autogluon:
                model_names.extend(["lightgbm","rf"])
            top_models = []
            if use_tqdm:
                for name in tqdm(model_names, desc="모델 학습 진행"):
                    model = create_model(name, verbose=False)
                    top_models.append(model)
            else:
                for name in model_names:
                    model = create_model(name, verbose=False)
                    top_models.append(model)
        else:
            top_models = compare_models(n_select=top_n, sort=sort_metric, turbo=True, verbose=True)
            if not isinstance(top_models, list):
                top_models = [top_models]

        blended = blend_models(estimator_list=top_models, optimize=sort_metric, choose_better=True, verbose=True)
        self.best_model = blended
        gc.collect()
        print(f"🏆 Blended model 생성 완료 (기준={sort_metric})")
        return self.best_model
    # eof -----------------------------------

    # save_metrics ##########################
    def save_metrics(self, model_name=None):
        """성능 지표 저장"""
        if self.best_model is None:
            raise ValueError("모델이 없습니다.")
        pred_df = predict_model(self.best_model, data=self.train_vectorized.copy())
        y_log_true = self.train["price"].values
        y_log_pred = pred_df["prediction_label"].values
        y_true = np.expm1(y_log_true)
        y_pred = np.expm1(y_log_pred)
        r2 = r2_score(y_true, y_pred)
        rmse = mean_squared_error(y_true, y_pred, squared=False)
        mae = mean_absolute_error(y_true, y_pred)
        self.metrics = {"R2": round(r2,4),"RMSE": round(rmse,4),"MAE": round(mae,4)}
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        if model_name is None:
            model_name = str(self.best_model).split("(")[0]
        file_path = os.path.join(self.results_dir, f"{model_name}_metrics_{timestamp}.json")
        with open(file_path,"w") as f:
            json.dump(self.metrics,f,indent=4)
        print(f"💾 Metrics 저장 완료: {file_path}")
        print(f"   - R² = {self.metrics['R2']}")
        print(f"   - RMSE = ${self.metrics['RMSE']:.2f}")
        print(f"   - MAE = ${self.metrics['MAE']:.2f}")
    # eof -----------------------------------

    # predict_test ##########################
    def predict_test(self, submission_file="submission.csv"):
        """테스트 예측 및 제출 파일 생성"""
        if self.best_model is None:
            raise ValueError("먼저 find_and_blend_models()를 실행하세요.")
        predictions = predict_model(self.best_model, data=self.test_vectorized.copy())
        price_log_pred = predictions["prediction_label"].values
        price_pred = np.expm1(price_log_pred)
        submission = pd.DataFrame({"test_id": self.test["test_id"], "price": price_pred})
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        submission_path = os.path.join(self.results_dir, f"{timestamp}_{submission_file}")
        submission.to_csv(submission_path, index=False)
        print(f"💾 Submission 저장 완료: {submission_path}")
        return submission
      
      
    # save_vectorized #######################
    def save_vectorized(self, method="tfidf"):
        """벡터화된 데이터 저장"""
        os.makedirs(self.results_dir, exist_ok=True)
        train_path = os.path.join(self.results_dir, f"vectorized_{method}_train.pkl")
        test_path = os.path.join(self.results_dir, f"vectorized_{method}_test.pkl")
        self.train_vectorized.to_pickle(train_path)
        self.test_vectorized.to_pickle(test_path)
        print(f"💾 {method} 벡터화 결과 저장 완료: {train_path}, {test_path}")
    # eof -----------------------------------
    
    # load_vectorized #######################
    def load_vectorized(self, method="tfidf"):
        """저장된 벡터화 데이터 불러오기"""
        train_path = os.path.join(self.results_dir, f"vectorized_{method}_train.pkl")
        test_path = os.path.join(self.results_dir, f"vectorized_{method}_test.pkl")
        if os.path.exists(train_path) and os.path.exists(test_path):
            self.train_vectorized = pd.read_pickle(train_path)
            self.test_vectorized = pd.read_pickle(test_path)
            print(f"📂 {method} 벡터화 결과 불러오기 완료")
            return True
        else:
            print(f"⚠️ {method} 벡터화 결과 파일이 없습니다. 새로 벡터화해야 합니다.")
            return False
    # eof -----------------------------------
    
    
    
          
    # eof -----------------------------------        